SECTION 1 — Environment Setup (Original)

In [ ]:
# ============================
# CELL 1 — Force Fix for NumPy (The Hammer)
# ============================

!pip uninstall -y numpy
!pip uninstall -y ultralytics
!pip install "numpy==1.23.5" ultralytics --no-warn-conflicts

import os
import torch
import yaml
import warnings
import numpy as np

warnings.filterwarnings('ignore')
os.environ['WANDB_DISABLED'] = 'true' 

print(f"NumPy Version: {np.__version__}")
print(f"Torch Version: {torch.__version__}")
print(f"GPUs Detected: {torch.cuda.device_count()}")

torch.manual_seed(42)


SECTION 2 — Dataset Configuration (Original)

In [ ]:
# ============================
# CELL 2 — Create Data Config (YAML)
# ============================

DATA_ROOT = '/kaggle/input/sku110dataset-resized640/v1.0/v1.0'

data_config = {
    'path': DATA_ROOT,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 1,
    'names': ['sku_product']
}

yaml_path = 'sku110_rtdetr.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print(open(yaml_path).read())


SECTION 3 — Dataset Preparation & Analysis

3.1 — Dataset Statistics

In [ ]:
# ============================
# CELL 3 — Dataset Statistics
# ============================

from pathlib import Path
import cv2
from collections import Counter

def dataset_stats():
    print("=== Dataset Info ===")
    for split in ['train', 'val', 'test']:
        img_dir = Path(DATA_ROOT)/'images'/split
        imgs = list(img_dir.glob("*.jpg"))
        print(f"{split}: {len(imgs)} images")

    # resolution check on first 20 images
    res = Counter()
    train_imgs = list((Path(DATA_ROOT)/'images'/'train').glob("*.jpg"))
    for img in train_imgs[:20]:
        im = cv2.imread(str(img))
        if im is not None:
            h, w = im.shape[:2]
            res[(w, h)] += 1

    print("\nSample Resolutions (First 20 Train Images):")
    print(res)

dataset_stats()


3.2 — Verify Resizing

In [ ]:
# ============================
# CELL 4 — Verify All Images are 640×640
# ============================

def verify_resizing(split):
    img_dir = Path(DATA_ROOT)/'images'/split
    imgs = list(img_dir.glob("*.jpg"))
    wrong = []

    for p in imgs:
        img = cv2.imread(str(p))
        if img is None: continue
        h, w = img.shape[:2]
        if h != 640 or w != 640:
            wrong.append((p.name, (w, h)))

    print(f"\n{split.upper()} — Total: {len(imgs)}")
    if len(wrong) == 0:
        print("✓ All images are 640×640")
    else:
        print("⚠ Wrong Resolutions:", wrong[:10])

for s in ['train','val','test']:
    verify_resizing(s)


3.3 — Dataset Cleaning

In [ ]:
# ============================
# CELL 5 — Dataset Integrity Checking
# ============================

from tqdm.auto import tqdm

def check_integrity(split):
    img_dir = Path(DATA_ROOT)/'images'/split
    lbl_dir = Path(DATA_ROOT)/'labels'/split

    imgs = list(img_dir.glob("*.jpg"))
    missing = []
    corrupt = []

    for p in tqdm(imgs, desc=f"Checking {split}"):
        if not (lbl_dir/(p.stem + ".txt")).exists():
            missing.append(p.name)

        if cv2.imread(str(p)) is None:
            corrupt.append(p.name)

    print(f"\n=== {split.upper()} Integrity ===")
    print(f"Missing labels: {len(missing)}")
    print(f"Corrupted images: {len(corrupt)}")

for s in ['train','val','test']:
    check_integrity(s)


3.5 — Bounding Box Stats

In [ ]:
# ============================
# CELL 7 — Bounding Box Stats
# ============================

import numpy as np

def bbox_stats(split='train'):
    lbl_dir = Path(DATA_ROOT)/'labels'/split
    widths=[]
    heights=[]

    for p in lbl_dir.glob("*.txt"):
        for line in open(p):
            cls,cx,cy,w,h = map(float,line.split())
            widths.append(w)
            heights.append(h)

    widths=np.array(widths)
    heights=np.array(heights)

    print(f"\n=== {split.upper()} BBOX STATS ===")
    print("Avg width:", widths.mean())
    print("Avg height:", heights.mean())
    print("Range width:", widths.min(), "→", widths.max())
    print("Range height:", heights.min(), "→", heights.max())

bbox_stats("train")
bbox_stats("val")


SECTION 4 — Load RT-DETR (Original)

In [ ]:
# ============================
# CELL 8 — Load RT-DETR Model (Original)
# ============================

from ultralytics import RTDETR
model = RTDETR('rtdetr-l.pt')

print("Model Loaded Successfully: RT-DETR-L")


SECTION 5 — Hyperparameter Tuning 

5.1 Define Experiments

In [ ]:
# ============================
# CELL 9 — Define 3 Experiments (10 Epochs)
# ============================

hparam_experiments = [
    {"name": "EXP_LR1e-4_B8_ADAMW", "batch": 8, "lr0": 1e-4, "optimizer": "AdamW"},
    {"name": "EXP_LR5e-4_B8_ADAMW", "batch": 8, "lr0": 5e-4, "optimizer": "AdamW"},
    {"name": "EXP_LR1e-4_B16_SGD",  "batch": 16, "lr0": 1e-4, "optimizer": "SGD"},
]

hparam_experiments


5.2 Run All Experiments (10 Epochs Each)

In [ ]:
# ============================
# CELL 10 — Run All Experiments (10 Epochs Each)
# ============================

from ultralytics import RTDETR

def run_hparam_exp(cfg):


    # RELOAD MODEL EVERY EXPERIMENT
    fresh_model = RTDETR("rtdetr-l.pt")
    print("\n========================================")
    print(f" RUNNING EXPERIMENT: {cfg['name']}")
    print("----------------------------------------")
    print(f" batch     = {cfg['batch']}")
    print(f" lr0       = {cfg['lr0']}")
    print(f" optimizer = {cfg['optimizer']}")
    print(f" epochs    = 10")
    print(f" imgsz     = 640")
    print("========================================\n")
    return fresh_model.train(
        data='sku110_rtdetr.yaml',
        epochs=10,               
        imgsz=640,
        batch=cfg["batch"],
        lr0=cfg["lr0"],
        optimizer=cfg["optimizer"],
        project='HParam_Experiments',
        name=cfg["name"],
        device=[0,1],
        cache=False,
        rect=True,
        val=True,
        verbose=True,
        exist_ok=True
    )

# Run All Experiments
for exp in hparam_experiments:
    run_hparam_exp(exp)

print("All 3 experiments completed (10 epochs each).")


5.3 Select Best Experiment

In [ ]:
# ============================
# CELL 11 — Select BEST Experiment Based on mAP50-95
# ============================

import pandas as pd

def get_map(exp_name):
    path = f"/kaggle/working/HParam_Experiments/{exp_name}/results.csv"
    df = pd.read_csv(path)
    map_cols = [c for c in df.columns if "mAP" in c]
    return df[map_cols[-1]].max()

results_summary = {}
for exp in hparam_experiments:
    name = exp["name"]
    results_summary[name] = get_map(name)
    print(f"{name} → mAP50-95: {results_summary[name]}")

best_exp = max(results_summary, key=results_summary.get)

print("\n=======================================")
print(f" BEST EXPERIMENT = {best_exp}")
print("=======================================")


5.4 CELL — FULL VISUALIZATION SUITE

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_full_training_results(csv_path, title="Training Visualization"):
    df = pd.read_csv(csv_path)
    epochs = df["epoch"]

    # Dynamically detect available loss columns
    train_loss_cols = [c for c in df.columns if "train" in c and "loss" in c]
    val_loss_cols = [c for c in df.columns if "val" in c and "loss" in c]

    # Dynamically detect precision, recall, mAP
    prec_cols = [c for c in df.columns if "precision" in c.lower()]
    recall_cols = [c for c in df.columns if "recall" in c.lower()]
    map50_cols = [c for c in df.columns if "map50" in c.lower()]
    map5095_cols = [c for c in df.columns if "map50-95" in c.lower() or "mAP50-95" in c]

    # detect LR
    lr_cols = [c for c in df.columns if "lr" in c.lower()]

    plt.figure(figsize=(20, 16))

    # -------------------------
    # 1) Training vs Validation LOSS
    # -------------------------
    plt.subplot(3, 2, 1)
    for col in train_loss_cols:
        plt.plot(epochs, df[col], label=f"Train {col.split('/')[-1]}")
    for col in val_loss_cols:
        plt.plot(epochs, df[col], label=f"Val {col.split('/')[-1]}")
    plt.title(f"{title} — Loss Curves")
    plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 2) Precision
    # -------------------------
    if prec_cols:
        plt.subplot(3, 2, 2)
        plt.plot(epochs, df[prec_cols[0]], label="Precision")
        plt.title(f"{title} — Precision")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 3) Recall
    # -------------------------
    if recall_cols:
        plt.subplot(3, 2, 3)
        plt.plot(epochs, df[recall_cols[0]], label="Recall")
        plt.title(f"{title} — Recall")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 4) mAP50
    # -------------------------
    if map50_cols:
        plt.subplot(3, 2, 4)
        plt.plot(epochs, df[map50_cols[0]], label="mAP50")
        plt.title(f"{title} — mAP50")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 5) mAP50-95
    # -------------------------
    if map5095_cols:
        plt.subplot(3, 2, 5)
        plt.plot(epochs, df[map5095_cols[0]], label="mAP50-95")
        plt.title(f"{title} — mAP50-95")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 6) Learning Rate
    # -------------------------
    if lr_cols:
        plt.subplot(3, 2, 6)
        plt.plot(epochs, df[lr_cols[0]], label="Learning Rate")
        plt.title(f"{title} — LR Curve")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    plt.tight_layout()
    plt.show()


CELL — Visualization for ALL 3 Experiments

In [ ]:
print("=== VISUALIZATION FOR ALL EXPERIMENTS ===")

for exp in hparam_experiments:
    csv_path = f"/kaggle/working/HParam_Experiments/{exp['name']}/results.csv"
    print(f"\n### Visualizing {exp['name']} ###")
    plot_full_training_results(csv_path, exp['name'])


SECTION 6 — Retrain BEST Experiment (25 Epochs)

In [ ]:
# ============================
# CELL 13 — Retrain BEST Experiment for 25 Epochs (Main Model)
# ============================

best_cfg = [e for e in hparam_experiments if e["name"] == best_exp][0]

print("\n=======================================")
print(f" Retraining BEST experiment ({best_exp}) for 25 EPOCHS")
print("=======================================\n")

main_model = model.train(
    data='sku110_rtdetr.yaml',
    epochs=25,               # <-- 25 EPOCHS FINAL TRAINING
    imgsz=640,
    batch=best_cfg["batch"],
    lr0=best_cfg["lr0"],
    optimizer=best_cfg["optimizer"],
    project='Final_Main_Model',
    name='RT-DETR_Main',
    device=[0,1],
    cache=False,
    val=True,
    verbose=True,
    exist_ok=True
)


SECTION 6.2 — Full Visualization Suite for Final Training

In [ ]:
# ============================
# CELL 14 — Full Training Visualization (Loss + mAP + LR)
# ============================

import pandas as pd
import matplotlib.pyplot as plt

def plot_full_training_results(csv_path, title="Training Visualization"):
    df = pd.read_csv(csv_path)
    epochs = df["epoch"]

    # Dynamically detect available loss columns
    train_loss_cols = [c for c in df.columns if "train" in c and "loss" in c]
    val_loss_cols = [c for c in df.columns if "val" in c and "loss" in c]

    # Dynamically detect precision, recall, mAP
    prec_cols = [c for c in df.columns if "precision" in c.lower()]
    recall_cols = [c for c in df.columns if "recall" in c.lower()]
    map50_cols = [c for c in df.columns if "map50" in c.lower()]
    map5095_cols = [c for c in df.columns if "map50-95" in c.lower() or "mAP50-95" in c]

    # detect LR
    lr_cols = [c for c in df.columns if "lr" in c.lower()]

    plt.figure(figsize=(20, 16))

    # -------------------------
    # 1) Training vs Validation LOSS
    # -------------------------
    plt.subplot(3, 2, 1)
    for col in train_loss_cols:
        plt.plot(epochs, df[col], label=f"Train {col.split('/')[-1]}")
    for col in val_loss_cols:
        plt.plot(epochs, df[col], label=f"Val {col.split('/')[-1]}")
    plt.title(f"{title} — Loss Curves")
    plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 2) Precision
    # -------------------------
    if prec_cols:
        plt.subplot(3, 2, 2)
        plt.plot(epochs, df[prec_cols[0]], label="Precision")
        plt.title(f"{title} — Precision")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 3) Recall
    # -------------------------
    if recall_cols:
        plt.subplot(3, 2, 3)
        plt.plot(epochs, df[recall_cols[0]], label="Recall")
        plt.title(f"{title} — Recall")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 4) mAP50
    # -------------------------
    if map50_cols:
        plt.subplot(3, 2, 4)
        plt.plot(epochs, df[map50_cols[0]], label="mAP50")
        plt.title(f"{title} — mAP50")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 5) mAP50-95
    # -------------------------
    if map5095_cols:
        plt.subplot(3, 2, 5)
        plt.plot(epochs, df[map5095_cols[0]], label="mAP50-95")
        plt.title(f"{title} — mAP50-95")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 6) Learning Rate
    # -------------------------
    if lr_cols:
        plt.subplot(3, 2, 6)
        plt.plot(epochs, df[lr_cols[0]], label="Learning Rate")
        plt.title(f"{title} — LR Curve")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    plt.tight_layout()
    plt.show()

# Run Visualization for Final Model
final_csv = "/kaggle/working/Final_Main_Model/RT-DETR_Main/results.csv"
plot_full_training_results(final_csv, "RT-DETR FINAL MODEL")


SECTION 7 — Load MAIN MODEL

In [ ]:
# ============================
# CELL 14 — Load FINAL MAIN MODEL
# ============================

from ultralytics import RTDETR

final_model_path = "/kaggle/working/Final_Main_Model/RT-DETR_Main/weights/best.pt"
final_model = RTDETR(final_model_path)

print("\n=======================================")
print(" Final model loaded successfully!")
print("=======================================\n")


SECTION 8 — Evaluate MAIN MODEL

In [ ]:
# ============================
# CELL 15 — Final Test Evaluation (Metrics Only)
# ============================

final_metrics = final_model.val(
    data='sku110_rtdetr.yaml',
    split='test',
    device=[0,1]
)

print("\n=======================================")
print(" FINAL TEST METRICS")
print("=======================================\n")
print(f"mAP50       : {final_metrics.box.map50:.4f}")
print(f"mAP50-95    : {final_metrics.box.map:.4f}")
print(f"Precision   : {final_metrics.box.mp:.4f}")
print(f"Recall      : {final_metrics.box.mr:.4f}")

# F1 score computation
f1 = 2 * (final_metrics.box.mp * final_metrics.box.mr) / (final_metrics.box.mp + final_metrics.box.mr)
print(f"F1 Score    : {f1:.4f}")


In [ ]:
# ============================
# CELL 16 — Generate Evaluation Plots (PR, F1, Confidence curves)
# ============================

eval_results = final_model.val(
    data='sku110_rtdetr.yaml',
    split='test',
    save_json=True,
    save_hybrid=True,
    plots=True,   # <--- Generates PR, F1, Confidence curves
    device=[0,1]
)

print("\nEvaluation plots saved in:")
print(eval_results.save_dir)


In [ ]:
# ============================
# CELL 17 — Speed Test (Inference Latency)
# ============================

from pathlib import Path
import random

test_images = list((Path(DATA_ROOT)/"images"/"test").glob("*.jpg"))
test_img = random.choice(test_images)

speed_results = final_model.predict(
    source=str(test_img),
    device=0,
    conf=0.25,
    verbose=False
)

print("\n=======================================")
print(" MODEL SPEED TEST (ms)")
print("=======================================\n")
print(speed_results[0].speed)


In [ ]:
# ============================
# CELL 18 — Sample Predictions Visualization
# ============================

import matplotlib.pyplot as plt
import cv2
import numpy as np

sample_imgs = random.sample(test_images, 4)

plt.figure(figsize=(18,14))

for i, img_path in enumerate(sample_imgs, start=1):
    pred = final_model.predict(source=str(img_path), conf=0.25, device=0)[0]
    img = pred.plot()   # BGR

    plt.subplot(2,2,i)
    plt.imshow(img[..., ::-1])   # Convert BGR → RGB
    plt.title(f"Prediction: {img_path.name}")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ============================
# CELL 19 — Summary Table (Copy into Report)
# ============================

print("\n=======================================")
print(" SUMMARY TABLE — FINAL MODEL")
print("=======================================\n")

print(f"""
MODEL NAME                 : RT-DETR (Final Best Model)
TEST IMAGES                : {len(test_images)}
mAP50                      : {final_metrics.box.map50:.4f}
mAP50-95                   : {final_metrics.box.map:.4f}
Precision                  : {final_metrics.box.mp:.4f}
Recall                     : {final_metrics.box.mr:.4f}
F1 Score                   : {f1:.4f}

Inference Speed (Preprocess): {speed_results[0].speed['preprocess']:.2f} ms
Inference Speed (Model)     : {speed_results[0].speed['inference']:.2f} ms
""")


SECTION 10 — Learning Curve

In [ ]:
# ============================
# CELL 17 — Learning Curve (Final Model)
# ============================

df = pd.read_csv("/kaggle/working/Final_Main_Model/RT-DETR_Main/results.csv")
map_col = [c for c in df.columns if "mAP" in c][0]

plt.plot(df['epoch'], df[map_col])
plt.title("Learning Curve — Main Model")
plt.xlabel("Epoch")
plt.ylabel(map_col)
plt.grid()
plt.show()


SECTION 11 — Interface (Gradio)

In [ ]:
# ============================
# CELL 18 — Gradio Interface
# ============================

# !pip install gradio --quiet
import gradio as gr

def infer(image):
    preds = final_model.predict(source=image, conf=0.25, device=0)
    return preds[0].plot()[..., ::-1]

demo = gr.Interface(fn=infer, inputs="image", outputs="image")
# demo.launch(share=True)
